# 基本使用

所有结构化输出的基本模型都要继承BaseModel

## 大模型的初始化

In [ ]:
import os
from enum import Enum
from typing import Optional, Literal, List

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

"""
DeepSeek 的 deepseek-flash 默认开启了“思考模式”（thinking mode），而该模式与 LangChain with_structured_output 所使用的强制 tool_choice 参数不兼容。当思考模式激活时，API 只接受tool_choice="auto" 或 "none"，任何强制指定工具的选择都会被拒绝。
"""
model = init_chat_model(
    model="deepseek-flash",
    model_provider="deepseek",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    extra_body={"thinking": {"type": "disabled"}},  # 关闭思考模式
)

## 定义Pydantic模型
使用清晰的字段描述；没有描述，LLM可能格式错误

In [ ]:
from pydantic import BaseModel, Field, ValidationError


class Person(BaseModel):
    """人物描述"""
    name: str = Field(description="姓名")
    age: int = Field(description="年龄")
    occupation: str = Field(description="职业")

struct_model = model.with_structured_output(Person)
response = struct_model.invoke("张三是一名30岁的软件工程师")
print(response)
print(type(response))

In [ ]:
class MovieModel(BaseModel):
    """
    电影的详细信息
    """
    title: str = Field(description="电影标题")
    year: int = Field(description="电影上映年份")
    director: str = Field(description="导演")
    rating: float = Field(description="电影评分，满分十分")

model_with_structure = model.with_structured_output(MovieModel)
response = model_with_structure.invoke("给出盗梦空间的信息")
print(response)
print(type(response))

In [ ]:
class SentimentAnalysis(BaseModel):
    """情感分析结果"""
    sentiment: str = Field(description="情感倾向：positive/negative/neutral")
    confidence: float = Field(description="置信度，0-1之间")
    keywords: list[str] = Field(description="关键词列表")

structured_model = model.with_structured_output(SentimentAnalysis)
# 调用
text = "这个课程内容很实用，学到了很多知识，强烈推荐！"
result = structured_model.invoke(
f"分析以下文本的情感：\n{text}"
)
print(f"类型: {type(result)}") # <class 'SentimentAnalysis'>
print(f"情感: {result.sentiment}")
print(f"置信度: {result.confidence}")
print(f"关键词: {result.keywords}")

## 高级特性

In [ ]:
import os

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

"""
DeepSeek 的 deepseek-flash 默认开启了“思考模式”（thinking mode），而该模式与 LangChain with_structured_output 所使用的强制 tool_choice 参数不兼容。当思考模式激活时，API 只接受tool_choice="auto" 或 "none"，任何强制指定工具的选择都会被拒绝。
"""
model = init_chat_model(
    model="deepseek-flash",
    model_provider="deepseek",
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL,
    extra_body={"thinking": {"type": "disabled"}},  # 关闭思考模式
)

### 1）可选字段

In [ ]:
from pydantic import BaseModel, Field
from typing import Optional

class Person(BaseModel):
    """人物描述"""
    name: str = Field(description="姓名")
    age: Optional[int] = Field(description="年龄")
    occupation: str = Field(description="职业")

struct_model = model.with_structured_output(Person)
response = struct_model.invoke("张三是一名软件工程师")
print(response)
print(type(response))

### 2）默认值

In [ ]:
from pydantic import BaseModel, Field
from typing import Optional

class Person(BaseModel):
    """人物描述"""
    name: str = Field(default="aa", description="姓名")
    age: Optional[int] = Field(30, description="年龄")
    occupation: str = Field("developer", description="职业")

struct_model = model.with_structured_output(Person)
response = struct_model.invoke("张三今天请假了")
print(response)
print(type(response))

### 3）枚举类型

In [ ]:
from enum import Enum

class Priority(str, Enum):
    LOW = "low"
    MEDIUM = "medium"
    HIGH = "high"

class CustomerInfo(BaseModel):
    """客户信息"""
    name: str = Field(description="客户姓名")
    phone: str = Field(description="电话号码")
    email: Optional[str] = Field(description="邮箱")
    issue: str = Field(description="问题描述")
    urgency: Priority = Field(description="紧急程度")
    # 使用 Literal 直接限定字面量值
    #urgency: Literal["low", "medium", "high"] = Field(description="紧急程度")

structured_llm = model.with_structured_output(CustomerInfo)
conversation = """
客服: 您好，请问有什么可以帮助您？
客户: 我是王小明，电话 138-1234-5678，我的订单一直没发货，很着急！
客服: 好的，我帮您查一下
"""
result = structured_llm.invoke(f"从以下客服对话中提取客户信息：\n{conversation}")
print(result)
print("\n提取结果：")
print(f" 客户: {result.name}")
print(f" 电话: {result.phone}")
print(f" 邮箱: {result.email or '未提供'}")
print(f" 问题: {result.issue}")
print(f" 紧急程度: {result.urgency.value}")

### 4）列表提取

In [ ]:
from typing import List

class Invoice(BaseModel):
    """发票信息"""
    invoice_number: str = Field(description="发票号")
    date: str = Field(description="日期")
    total_amount: float = Field(description="总金额")
    items: List[str] = Field(description="商品")

structured_llm = model.with_structured_output(Invoice)
invoice_text = """
发票号: INV-2024-001
日期: 2024-01-15
总金额: 1299.00
商品: MacBook Pro, AppleCare+
"""
invoice = structured_llm.invoke(f"提取发票信息：{invoice_text}")
print(invoice)

### 5）嵌套结构

In [ ]:
class Actor(BaseModel):
    """演员信息"""
    name: str = Field(description="演员姓名")
    role: str = Field(description="饰演的角色")

class Movie(BaseModel):
    title: str = Field(description="电影标题")
    year: int = Field(description="上映年份")
    director: str = Field(description="导演")
    cast: List[Actor] = Field(description="演员列表") # 定义列表字段
    rating: float = Field(description="评分")

structured_llm = model.with_structured_output(Movie)
response = structured_llm.invoke("请介绍电影盗梦空间")
print(response)

### 6）限制条件

In [ ]:
from pydantic import ValidationError

class User(BaseModel):
    name: str = Field(min_length=2, max_length=20)
    age: int = Field(ge=0, le=150)
    email: str

try:
    user = User(name="张三", age=20, email="8888@qq.com")
    print(f"OK {user}")
except ValidationError as e:
    print(e)

try:
    user = User(name="李四", age=200, email="8888@qq.com")
    print(f"OK {user}")
except ValidationError as e:
    print(f"验证失败，{e.errors()[0]}")